# 12.02 OpenTelemetry — 벤더 중립 관측

[OpenTelemetry (OTel)](https://opentelemetry.io) 는 traces · metrics · logs 를 표준 포맷으로 수집·수출하는 **CNCF 프로젝트**다.
LangChain / LangGraph 실행을 OTel span 으로 변환하면, **하나의 계측 코드**로 Datadog · Grafana Tempo · Jaeger · Honeycomb · New Relic 등 **어떤 백엔드로도 라우팅**할 수 있다.

## 학습 목표

- `OTEL_EXPORTER_OTLP_ENDPOINT` + OTel SDK 로 trace 수출 파이프라인 구성
- **LangSmith OTEL exporter** — LangSmith 가 OTel 프로토콜로도 trace 수신 (최근 feature)
- **OpenLLMetry** — LLM 전용 semantic convention 과 자동 계측 라이브러리
- **Traceloop SDK** — OpenLLMetry 의 상위 래퍼. 한 줄 `Traceloop.init(...)` 으로 LangChain/OpenAI/Anthropic 자동 계측
- 로컬 **Jaeger** 컨테이너로 시각화
- **Datadog · Grafana Tempo** 프로덕션 송출 설정

## LangSmith / Langfuse 와의 관계

- `06_langsmith/` — LangSmith 전용 UI + LangChain 자동 계측 (제어권 LangChain Inc.)
- `12_observability/01_langfuse.ipynb` — Langfuse 자체 UI (self-host 가능)
- **이 노트북** — OTel 표준 경유로 **LangSmith 든 Langfuse 든 Datadog 이든** 라우팅 가능. 프로바이더 잠금 회피.

## 12.02.1 환경 설정

### 필요 패키지

```bash
uv pip install opentelemetry-sdk opentelemetry-api \
               opentelemetry-exporter-otlp \
               traceloop-sdk
```

- `opentelemetry-sdk` + `opentelemetry-api` — OTel 런타임 (Tracer / Span)
- `opentelemetry-exporter-otlp` — OTLP/gRPC + OTLP/HTTP exporter 번들
- `traceloop-sdk` — OpenLLMetry 프로젝트의 상위 래퍼 (자동 계측 포함)

> `openllmetry` 라는 단일 PyPI 패키지는 없다. 프로젝트 이름이 OpenLLMetry 이고, 실제 설치는 `traceloop-sdk` 또는 `opentelemetry-instrumentation-<provider>` 개별 패키지로 한다.

### 필요 환경변수 (하나라도)

- `OTEL_EXPORTER_OTLP_ENDPOINT` — OTLP collector 주소 (예: `http://localhost:4318` for HTTP, `http://localhost:4317` for gRPC)
- `OTEL_EXPORTER_OTLP_HEADERS` — 인증 헤더 (예: `api-key=...`) — 벤더 종속

이 노트북의 코드 셀 대부분은 로컬 collector 가 있을 때만 실제 송출되며, 없으면 ConsoleExporter 로 **stdout 에 span JSON 출력** 하는 경로로 동작한다.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# 이 노트북이 요구하는 키/서비스만 probe
# OTel 은 키 없이도 동작한다. Console 출력으로 fallback.
import opentelemetry
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.resources import Resource, SERVICE_NAME

otel_endpoint = os.environ.get("OTEL_EXPORTER_OTLP_ENDPOINT")
print("OTel SDK :", opentelemetry.__version__ if hasattr(opentelemetry, "__version__") else "(pkg only)")
print("OTLP endpoint :", otel_endpoint or "(미설정 → ConsoleExporter 로 fallback)")
print("OPENAI_API_KEY:", "set" if os.environ.get("OPENAI_API_KEY") else "missing (LLM 호출 셀은 스킵)")

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 미설정 — 하단 LLM 호출 셀이 필요"

### 기본 TracerProvider 설정

OTel 은 프로세스당 **한 번** 전역 `TracerProvider` 를 설정하고, 이후 라이브러리들이 `trace.get_tracer(__name__)` 로 공유한다.
아래 블록은 collector 가 있으면 `OTLPSpanExporter` 로, 없으면 `ConsoleSpanExporter` 로 보내는 **dual-path** 를 만든다.

In [ ]:
from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter

resource = Resource.create({SERVICE_NAME: "agent-notebooks"})
provider = TracerProvider(resource=resource)

if otel_endpoint:
    # HTTP 기반 OTLP (endpoint 에 4318 포트가 보통). gRPC 쓰려면 proto-grpc exporter 로 교체.
    from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
    exporter = OTLPSpanExporter(endpoint=f"{otel_endpoint.rstrip('/')}/v1/traces")
    print("→ OTLP HTTP exporter:", exporter._endpoint)
else:
    exporter = ConsoleSpanExporter()
    print("→ ConsoleSpanExporter (stdout)")

provider.add_span_processor(BatchSpanProcessor(exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("agent-notebooks.12_02_otel")
with tracer.start_as_current_span("smoke-test") as span:
    span.set_attribute("ping", True)
print("span 1 개 생성")

## 12.02.2 LangSmith OTEL exporter

최근 LangSmith 는 자체 SDK 외에 **OTLP/HTTP 엔드포인트**도 제공한다. 이미 OTel 로 수집 중인 조직은 collector 라우팅 규칙만 추가하면 LangSmith 로 보낼 수 있다.

### 엔드포인트

```
https://api.smith.langchain.com/otel   (traces 서브패스: /v1/traces)
```

### 필요 헤더

```
x-api-key: <LANGSMITH_API_KEY>
Langsmith-Project: <프로젝트명>
```

### 환경변수로 처리하기 (권장)

```dotenv
OTEL_EXPORTER_OTLP_ENDPOINT=https://api.smith.langchain.com/otel
OTEL_EXPORTER_OTLP_HEADERS="x-api-key=lsv2_pt_xxxx,Langsmith-Project=my-service"
```

SDK 는 이 환경변수만으로 자동 설정된다. 아래 셀은 **설정만 보여주고 실제 호출은 않는다** — 실수로 개인 키가 트레이스에 섞여 나가지 않도록.

In [ ]:
# 예시 — 실제로는 .env 에 넣고 OTel SDK 가 자동 픽업하게 둔다.
langsmith_otel_example = {
    "OTEL_EXPORTER_OTLP_ENDPOINT": "https://api.smith.langchain.com/otel",
    "OTEL_EXPORTER_OTLP_HEADERS": "x-api-key=${LANGSMITH_API_KEY},Langsmith-Project=agent-notebooks",
}
for k, v in langsmith_otel_example.items():
    print(f"{k} = {v}")

> LangSmith 는 여전히 **native SDK 경로가 권장**된다 (자동 계측이 더 촘촘하고 dataset/evaluator 연동이 native). OTel 경로는 **이미 OTel 파이프라인이 있는 조직**이 LangSmith 로 일부 트래픽을 미러링할 때 쓴다.

## 12.02.3 OpenLLMetry — LLM 전용 semantic convention

OTel 의 [semantic conventions](https://opentelemetry.io/docs/specs/semconv/) 에는 HTTP / DB / messaging 은 있지만 **LLM 은 여전히 experimental** 이다. [OpenLLMetry](https://www.traceloop.com/openllmetry) 가 그 공백을 채우는 오픈소스 표준.

### 주요 attribute

- `gen_ai.system` — `openai` / `anthropic` / `google` ...
- `gen_ai.request.model` — 요청한 모델 ID (예: `gpt-4.1-mini`)
- `gen_ai.response.model` — 실제 응답 모델 (프로바이더 rewrite 가능)
- `gen_ai.usage.prompt_tokens` / `completion_tokens` / `total_tokens`
- `gen_ai.operation.name` — `chat` / `completion` / `embedding` / `text_completion`
- `gen_ai.prompt.N.role` / `gen_ai.prompt.N.content` — 전체 프롬프트 덤프 (옵션)

`traceloop-sdk` 를 초기화하면 이 규약이 모든 LLM/vectorstore/framework 호출에 **자동 주입**된다.

In [ ]:
# OpenLLMetry 규약은 attribute 만 표준화하면 된다 — 수동으로도 만들 수 있다.
from opentelemetry import trace

tracer = trace.get_tracer("demo.openllmetry")
with tracer.start_as_current_span("chat-completion") as span:
    span.set_attribute("gen_ai.system", "openai")
    span.set_attribute("gen_ai.request.model", "gpt-4.1-mini")
    span.set_attribute("gen_ai.operation.name", "chat")
    span.set_attribute("gen_ai.usage.prompt_tokens", 128)
    span.set_attribute("gen_ai.usage.completion_tokens", 64)
    span.set_attribute("gen_ai.usage.total_tokens", 192)
print("span 에 gen_ai.* 규약 attribute 주입 완료")

## 12.02.4 Traceloop SDK — OpenLLMetry 상위 래퍼

수동 attribute 주입은 금방 귀찮아진다. `traceloop-sdk` 는 초기화 한 번으로 다음을 자동 계측한다.

- OpenAI / Anthropic / Google / Cohere / Mistral / Bedrock / Ollama ...
- LangChain / LangGraph / LlamaIndex / Haystack
- Pinecone / Chroma / Weaviate / Qdrant / Milvus
- HTTP clients (requests / httpx / aiohttp)

`Traceloop.init(app_name=..., exporter=...)` 시그니처가 핵심. `exporter` 를 **직접 주입**하면 traceloop cloud 가 아니라 **임의 OTel 백엔드**로 보낼 수 있다.

In [ ]:
from traceloop.sdk import Traceloop
from opentelemetry.sdk.trace.export import ConsoleSpanExporter

# disable_batch=True: 노트북에서 즉시 flush 되도록 (프로덕션에선 False=배치 권장)
Traceloop.init(
    app_name="agent-notebooks-otel-demo",
    exporter=ConsoleSpanExporter(),     # 실제로는 OTLPSpanExporter 로 교체
    disable_batch=True,
    telemetry_enabled=False,            # Traceloop 의 usage 수집 끄기
)
print("Traceloop 초기화 완료 — 이제 LLM / LangChain 호출이 자동 계측된다.")

### 자동 계측 확인 — LangChain agent 호출

`Traceloop.init(...)` 이후에 만든 LangChain 에이전트는 **코드 수정 없이** 각 LLM / tool 호출마다 OpenLLMetry 규약의 span 이 자동으로 찍힌다. 아래 실행 결과를 보면 `openai.chat` 계열 span 이 stdout 에 흘러나온다.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 반환한다."""
    return f"{city}: 맑음, 21°C"

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[get_weather],
    system_prompt="당신은 친절한 날씨 봇입니다.",
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "서울 날씨?"}]
})
print("\n최종 응답:", result["messages"][-1].content)

## 12.02.5 로컬 Jaeger 로 시각화

Jaeger all-in-one 은 collector + UI + in-memory 저장소를 한 컨테이너에 담은 **로컬 개발용 최소 구성**이다.

### 기동

```bash
docker run -d --name jaeger \
  -p 16686:16686 \
  -p 4317:4317  \
  -p 4318:4318  \
  -e COLLECTOR_OTLP_ENABLED=true \
  jaegertracing/all-in-one:latest
```

- `16686` — Web UI (http://localhost:16686)
- `4317` — OTLP gRPC
- `4318` — OTLP HTTP

### .env 전환

```dotenv
OTEL_EXPORTER_OTLP_ENDPOINT=http://localhost:4318
```

### 연결 probe

아래 셀은 Jaeger UI 루트에 GET 을 시도해 컨테이너 기동 여부를 확인한다. 컨테이너가 없으면 `requests.exceptions.ConnectionError` 대신 **timeout 짧게** 해서 `None` 으로 떨어뜨린다.

In [ ]:
import socket

def probe(host: str, port: int, timeout: float = 1.0) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(timeout)
        return s.connect_ex((host, port)) == 0

jaeger_ui   = probe("localhost", 16686)
jaeger_otlp = probe("localhost", 4318)
print(f"Jaeger UI   (16686) : {'UP' if jaeger_ui   else 'down'}")
print(f"Jaeger OTLP (4318)  : {'UP' if jaeger_otlp else 'down'}")

if jaeger_otlp:
    print("→ http://localhost:16686 에서 Service: 'agent-notebooks-otel-demo' 필터")
else:
    print("→ 컨테이너 미기동. 위 docker 명령으로 띄운 뒤 재실행.")

### Jaeger UI 에서 보는 지점

1. 좌측 `Service` 드롭다운 → `agent-notebooks-otel-demo` 선택
2. `Operation` → `openai.chat` / `langchain.agent_executor` / `langchain.tool.get_weather` ...
3. trace 하나 클릭 → waterfall 에서 각 span 의 latency · `gen_ai.*` attribute 확인
4. `Compare` 탭 — 두 trace 구조 diff (A/B 실험에 유용)

## 12.02.6 프로덕션 송출 — Datadog · Grafana Tempo

같은 OTel SDK 로 백엔드만 갈아끼우는 패턴. 보통 **OTel Collector** 를 중간에 두고 (`otelcol` 바이너리), 애플리케이션은 collector 로만 보내고 collector 가 벤더별 exporter 로 팬아웃한다. 백엔드가 바뀌어도 앱 재배포가 불필요하다.

### Datadog

두 가지 경로.

1. **Datadog Agent 가 OTLP 수신** (권장, agent 7.42+)
    ```dotenv
    OTEL_EXPORTER_OTLP_ENDPOINT=http://datadog-agent:4318
    ```
    agent 의 `datadog.yaml` 에 `otlp_config.receiver.protocols.http.endpoint: 0.0.0.0:4318` 추가.

2. **Datadog OTLP Intake 직접 송출** (에이전트 없는 환경)
    ```dotenv
    OTEL_EXPORTER_OTLP_ENDPOINT=https://trace.agent.datadoghq.com
    OTEL_EXPORTER_OTLP_HEADERS="DD-API-KEY=${DD_API_KEY}"
    ```

### Grafana Tempo (+ Cloud)

```dotenv
OTEL_EXPORTER_OTLP_ENDPOINT=https://tempo-prod-XX-prod-us-central-0.grafana.net:443
OTEL_EXPORTER_OTLP_HEADERS="Authorization=Basic ${BASIC_AUTH_B64}"
```

- `BASIC_AUTH_B64` = base64(`<instance_id>:<api_token>`)
- self-host Tempo 는 `http://tempo:4318` (Kubernetes 의 경우 Service 이름)

### Collector 경유 팬아웃 예 (`otelcol-contrib.yaml`)

```yaml
receivers:
  otlp:
    protocols:
      http:
      grpc:

exporters:
  datadog:
    api:
      key: ${env:DD_API_KEY}
  otlphttp/langsmith:
    endpoint: https://api.smith.langchain.com/otel
    headers:
      x-api-key: ${env:LANGSMITH_API_KEY}
  otlp/tempo:
    endpoint: tempo:4317
    tls: { insecure: true }

service:
  pipelines:
    traces:
      receivers: [otlp]
      exporters: [datadog, otlphttp/langsmith, otlp/tempo]
```

→ 앱은 `otelcol` 하나만 바라보고, collector 가 Datadog + LangSmith + Tempo **삼중 송출**.
백엔드 추가/삭제는 YAML 한 줄로 끝난다 — **이것이 OTel 을 쓰는 이유**.

### 종료 전 flush

`BatchSpanProcessor` 는 기본 5 초 배치라 노트북/스크립트가 종료되기 전 강제 flush 하지 않으면 마지막 span 이 누락된다.

In [ ]:
provider.shutdown()    # 내부에서 flush 후 processor 닫음
print("TracerProvider shutdown — 모든 span 송출 완료")

## 체크리스트

- [ ] `opentelemetry-sdk` + `opentelemetry-exporter-otlp` 설치
- [ ] 프로세스당 `TracerProvider` 는 **한 번만** 설정
- [ ] `OTEL_EXPORTER_OTLP_ENDPOINT` 가 HTTP(4318) 인지 gRPC(4317) 인지 확인 — exporter 종류가 달라짐
- [ ] LangSmith 로도 보내려면 `https://api.smith.langchain.com/otel` + `x-api-key` 헤더
- [ ] 자동 계측은 **`Traceloop.init(...)` 한 줄** — LangChain · OpenAI · vectorstore 전부 `gen_ai.*` attribute 자동 주입
- [ ] 로컬 개발은 Jaeger all-in-one (`docker run ... jaegertracing/all-in-one`)
- [ ] 프로덕션은 **OTel Collector 를 중간에 두는** 팬아웃 설계 권장
- [ ] 종료 전 `TracerProvider.shutdown()` 으로 flush

## 다음

- `06_langsmith/05_production_monitoring.ipynb` — LangSmith 쪽 프로덕션 모니터링
- `01_langfuse.ipynb` — Langfuse self-host (내부가 OTel 기반, 동일 규약)

## 참고

- OpenTelemetry 공식: https://opentelemetry.io/docs/languages/python/
- OTel Semantic Conventions for Gen AI: https://opentelemetry.io/docs/specs/semconv/gen-ai/
- OpenLLMetry: https://www.traceloop.com/docs/openllmetry/introduction
- Traceloop SDK: https://github.com/traceloop/openllmetry
- LangSmith OTLP ingest: https://docs.smith.langchain.com/observability/how_to_guides/trace_with_opentelemetry
- Datadog OTel: https://docs.datadoghq.com/opentelemetry/
- Grafana Tempo: https://grafana.com/docs/tempo/latest/
- Jaeger: https://www.jaegertracing.io/docs/latest/getting-started/